# 03 · Detections on video

Runs a trained model on consecutive frames of one clip and renders an MP4:
- **predictions** with score ≥ `SCORE_THR`: box in the class colour, class name and score;
- **missed objects** (ground truth with no same-class prediction at IoU ≥ 0.5): red box;
- **ignored regions**: translucent magenta (detections there are not scored).

Missed objects are only drawn for `val`/`train` clips, which have labels. `test` clips show predictions only.

## Setup

In [ ]:
import json
import sys
from collections import defaultdict
from pathlib import Path

import cv2
import imageio.v2 as imageio
import numpy as np
import torch
from IPython.display import Video, display
from torchvision.ops import box_iou
from tqdm.auto import tqdm

sys.path.insert(0, str(Path.cwd()))
from common.data import CLASSES, to_model_input
from common.frcnn import load_faster_rcnn_run

DATA_ROOT = Path("../multiple-object-video-detection-hw-2-ucu-ai-2026")
RUN_DIR = Path("../runs/faster_rcnn/baseline")      # which trained model to show
SPLIT = "val"                                       # "val", "train" or "test"
CLIP = "uav0000137_00458_v"
FIRST_FRAME, N_FRAMES = 1, 150                      # consecutive frames to render
SCORE_THR = 0.4                                     # only draw confident predictions
FPS = 25
OUT_WIDTH = 1280                                    # video width (None = original size)
OUT_DIR = Path("../outputs/videos")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

COLORS = [(0, 255, 0), (255, 255, 0), (0, 165, 255), (255, 0, 0),
          (0, 0, 255), (0, 255, 255), (128, 0, 255), (255, 128, 128)]  # BGR, one per class
MISSED, IGNORE = (0, 0, 255), (255, 0, 255)

## Load model, frames and ground truth

In [ ]:
model = load_faster_rcnn_run(RUN_DIR, DEVICE)

img_dir = DATA_ROOT / SPLIT / "images" / CLIP
frames = sorted(img_dir.glob("*.jpg"))
frames = [f for f in frames if int(f.stem) >= FIRST_FRAME][:N_FRAMES]
print(f"{CLIP}: rendering {len(frames)} frames from {frames[0].stem}")

# Ground truth and ignored regions per frame (labelled splits only)
gt, ignore = defaultdict(list), defaultdict(list)
ann_file = DATA_ROOT / SPLIT / "annotations_coco.json"
if SPLIT != "test" and ann_file.exists():
    coco = json.loads(ann_file.read_text())
    frame_of = {im["id"]: im["frame_id"] for im in coco["images"] if im["video"] == CLIP}
    for a in coco["annotations"]:
        if a["image_id"] in frame_of:
            x, y, w, h = a["bbox"]
            (ignore if a["iscrowd"] else gt)[frame_of[a["image_id"]]].append((x, y, x + w, y + h, a["category_id"]))
    del coco

## Draw one frame

In [ ]:
def draw(img, out, frame_id):
    # Ignored regions: translucent fill
    overlay = img.copy()
    for x1, y1, x2, y2, _ in ignore.get(frame_id, []):
        cv2.rectangle(overlay, (int(x1), int(y1)), (int(x2), int(y2)), IGNORE, -1)
    img = cv2.addWeighted(overlay, 0.25, img, 0.75, 0)

    keep = out["scores"] >= SCORE_THR
    boxes, labels, scores = out["boxes"][keep].cpu(), out["labels"][keep].cpu(), out["scores"][keep].cpu()

    # Missed ground truth: no same-class prediction (score >= SCORE_THR) with IoU >= 0.5
    n_missed = 0
    g = gt.get(frame_id, [])
    if g:
        gb = torch.tensor([b[:4] for b in g], dtype=torch.float32)
        gc = torch.tensor([b[4] for b in g])
        if len(boxes):
            iou = box_iou(gb, boxes) * (gc[:, None] == labels[None, :])
            found = iou.max(1).values >= 0.5
        else:
            found = torch.zeros(len(g), dtype=torch.bool)
        for (x1, y1, x2, y2, _), ok in zip(g, found.tolist()):
            if not ok:
                n_missed += 1
                cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), MISSED, 2)

    # Predictions
    for (x1, y1, x2, y2), c, s in zip(boxes.tolist(), labels.tolist(), scores.tolist()):
        color = COLORS[c - 1]
        cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
        cv2.putText(img, f"{CLASSES[c - 1]} {s:.2f}", (int(x1), max(int(y1) - 4, 12)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)

    header = f"{RUN_DIR.name} | {CLIP} frame {frame_id} | {len(boxes)} detections (score >= {SCORE_THR})"
    if g:
        header += f" | {len(g)} objects, {n_missed} missed (red)"
    cv2.rectangle(img, (0, 0), (img.shape[1], 36), (0, 0, 0), -1)
    cv2.putText(img, header, (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
    return img, len(g), n_missed

## Render the video

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
out_path = OUT_DIR / f"pred_{RUN_DIR.name}_{CLIP}_{FIRST_FRAME}.mp4"
total_gt = total_missed = 0

with torch.no_grad(), imageio.get_writer(out_path, fps=FPS, codec="libx264", quality=7, macro_block_size=1) as writer:
    for f in tqdm(frames, desc="frames"):
        img = cv2.imread(str(f))
        rgb = torch.from_numpy(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)).permute(2, 0, 1)
        out = model(to_model_input([rgb], DEVICE))[0]
        vis, n_gt, n_missed = draw(img, out, int(f.stem))
        total_gt, total_missed = total_gt + n_gt, total_missed + n_missed
        if OUT_WIDTH and vis.shape[1] > OUT_WIDTH:
            vis = cv2.resize(vis, (OUT_WIDTH, int(vis.shape[0] * OUT_WIDTH / vis.shape[1]) // 2 * 2))
        writer.append_data(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))

print(f"Saved {out_path}")
if total_gt:
    print(f"Missed {total_missed} of {total_gt} labelled objects ({total_missed / total_gt:.0%}) at score >= {SCORE_THR}")

Plays inline (embedded: clear this output before committing), or open the MP4 from `outputs/videos/`.

In [ ]:
display(Video(str(out_path), embed=True, width=960))